# Simplest Semantic Segmentation with Pretrained weights

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load a pre-trained DeepLab model from TensorFlow Hub
# Explore different DeepLab models at https://tfhub.dev/s?q=deeplab
model_url = "https://tfhub.dev/google/deeplabv3_mnv2_dm05_coco/1"
segmentation_model = hub.load(model_url)

In [ ]:
# Load an image using OpenCV
image_path = "path/to/your/image.jpg"  # Replace with the path to your image
img = cv2.imread(image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
original_size = img.shape[:2][::-1]  # (width, height)

In [ ]:
# Resize the image to the model's expected input size
target_size = (512, 512)  # Common for DeepLab models
resized_img = cv2.resize(img, target_size)

In [ ]:
# Normalize the image (common preprocessing for these models)
normalized_img = resized_img / 255.0
input_tensor = tf.expand_dims(normalized_img, 0)  # Add batch dimension

In [ ]:
# Perform segmentation
output = segmentation_model(input_tensor)['semantic_segmentation'][0]
segmentation_mask = tf.argmax(output, axis=-1)
segmentation_mask = segmentation_mask.numpy().astype(np.uint8)

In [ ]:
# Resize the segmentation mask back to the original image size
segmentation_mask_resized = cv2.resize(segmentation_mask, original_size, interpolation=cv2.INTER_NEAREST)

In [ ]:
# Define a color map for the segmentation classes (common for COCO)
# You might need to adjust this based on the model
LABEL_COLORS = np.array([
    [0, 0, 0],        # 0: background
    [128, 0, 0],      # 1: aeroplane
    [0, 128, 0],      # 2: bicycle
    [128, 128, 0],    # 3: bird
    [0, 0, 128],      # 4: boat
    [128, 0, 128],    # 5: bottle
    [0, 128, 128],    # 6: bus
    [128, 128, 128],  # 7: car
    [64, 0, 0],       # 8: cat
    [192, 0, 0],      # 9: chair
    [64, 128, 0],     # 10: cow
    [192, 128, 0],    # 11: dining table
    [64, 0, 128],     # 12: dog
    [192, 0, 128],    # 13: horse
    [64, 128, 128],   # 14: motorbike
    [192, 128, 128],  # 15: person
    [0, 64, 0],       # 16: potted plant
    [128, 64, 0],     # 17: sheep
    [0, 192, 0],      # 18: sofa
    [128, 192, 0],    # 19: train
    [0, 64, 128],     # 20: tv/monitor
    # ... add more colors for other classes if needed
])

In [ ]:
# Create a colored segmentation mask
colored_mask = LABEL_COLORS[segmentation_mask_resized]

# Overlay the colored mask on the original image
alpha = 0.5
masked_image = cv2.addWeighted(img, 1 - alpha, colored_mask, alpha, 0)

# Display the results
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title('Original Image')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(masked_image)
plt.title('Segmentation Mask')
plt.axis('off')
plt.show()